# Batch-associated differences: a complete workflow

This notebook walks through one complete analysis with `neuro_rave`:

1. **before any correction**, measure how well the domain (batch, setup, session) can be predicted from
   held-out responses with `evaluation.assess_batch_effects`;
2. use biological labels, where you have them, to see whether the differences are *within* shared
   cell classes or come from different class *composition*;
3. decide what the evidence does and does not justify;
4. if you correct, fit `RAVE` (and optionally `RAVEPlus` or the experimental `VAEDANN`) with a
   zero-adversary control on training recordings only;
5. compare raw data and representations on the **same held-out recordings and splits**, together with
   reconstruction and biological prediction.

All data here are **synthetic** (generated in this notebook), so it runs on a laptop CPU in a few minutes.
One clearly marked cell is where you substitute your own data.

### When should I use this?

- **Before correction**, to see whether and how strongly your datasets differ.
- **After correction**, with the same split and probe protocol, on recordings the model did not see.
- **When comparing preprocessing or model choices**, on validation data, never on the final test set.
- **When correction may not be justified**: if differences follow biology (different cell populations,
  compartments or conditions), or if class and domain cannot be separated, a correction may remove the
  signal you care about.

**What the diagnostic answers:** how well domain identity can be predicted from held-out responses
under your sampling and split design. It does not say whether a difference is technical or
biological, whether correction is necessary, or that a lower score means a better representation. A
score near chance does not prove that there are no differences. No significance test is performed.

In [ ]:
import sys

import matplotlib.pyplot as plt
import numpy as np
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression

import neuro_rave
from neuro_rave import RAVE, VAEDANN, RAVEPlus
from neuro_rave import evaluation as ev

print("python    :", sys.executable)
print("neuro_rave:", neuro_rave.__version__, "from", neuro_rave.__file__)

# Optional sections (set to False to skip them).
RUN_RAVEPLUS = True  # needs labels y
RUN_VAEDANN = True  # experimental model

## 1. What the package expects

| Name | Shape / content | Role |
|---|---|---|
| `X` | `(samples, time)` or `(samples, time, channels)`: aligned responses on a shared time grid | the data |
| `domains` | one ID per sample (string or integer): the batch you ask about, e.g. setup or dataset | what the probe predicts |
| `groups` | one ID per sample: the unit that must stay together, e.g. recording, cell or animal | defines held-out data |
| `y` *(optional)* | biological class per sample, `None` where unknown | composition and within-class checks |

**Domains are not groups.** Domains are what you want to compare. Groups are the independent units
of your design, usually recordings nested within domains. The diagnostic holds out whole groups, so it
asks whether *new recordings* from known domains can be told apart. Using `groups=domains` is
rejected: holding out a whole domain would remove it from the probe's training data.

If each domain is a single session, holding out cells within it describes how separated known sessions
are, not whether unseen sessions could be identified. Many cells from one recording are not many
independent recordings.

## 2. Three synthetic examples: what the diagnostic can and cannot tell you

Three generated datasets share the same three response classes and the same recording design
(3 domains × 6 recordings × 40 cells):

- **A: shared populations, no imposed shift.** Domains differ only by chance.
- **B: same populations, technical shift.** Each domain distorts the responses with its own gain, offset
  and small time shift.
- **C: different composition, no technical shift.** The class proportions differ between domains.

The generator below is only for this notebook.

In [ ]:
TEMPLATES = {
    "transient": lambda t: np.exp(-(((t - 0.25) / 0.05) ** 2)),
    "sustained": lambda t: (
        1 / (1 + np.exp(-(t - 0.2) / 0.02)) - 1 / (1 + np.exp(-(t - 0.7) / 0.02))
    ),
    "oscillating": lambda t: np.sin(2 * np.pi * 4 * t) * np.exp(-(((t - 0.5) / 0.25) ** 2)),
}


def simulate(seed, domains, *, recordings=6, cells=40, shares=None, technical=None, labelled=0.7):
    """Synthetic two-channel responses: cells nested in recordings nested in domains."""
    rng = np.random.default_rng(seed)
    t = np.linspace(0, 1, 60)
    names = list(TEMPLATES)
    X, dom, grp, true = [], [], [], []
    for i, d in enumerate(domains):
        gain, offset, shift = technical[i] if technical else (1.0, 0.0, 0)
        for r in range(recordings):
            rec_gain = 1 + 0.1 * rng.normal()  # recordings differ a little within a domain
            for c in rng.choice(len(names), size=cells, p=shares[i] if shares else None):
                base = TEMPLATES[names[c]](t) * rng.uniform(0.7, 1.3)
                trace = np.stack([base, 0.6 * np.roll(base, 3)], axis=-1)
                trace = gain * rec_gain * np.roll(trace, shift, axis=0) + offset
                X.append(trace + 0.3 * rng.standard_normal(trace.shape))
                dom.append(d)
                grp.append(f"{d}/rec{r}")
                true.append(names[c])
    y = [c if rng.random() < labelled else None for c in true]
    return np.asarray(X), np.asarray(dom), np.asarray(grp), y, np.asarray(true)


RIGS = ["rig_A", "rig_B", "rig_C"]
SHIFT = [(1.0, 0.0, 0), (1.3, 0.25, 2), (0.75, -0.2, -2)]
SHARES = [[0.8, 0.15, 0.05], [0.34, 0.33, 0.33], [0.05, 0.15, 0.8]]
examples = {
    "A: no shift": simulate(1, RIGS),
    "B: technical shift": simulate(2, RIGS, technical=SHIFT),
    "C: composition differs": simulate(3, RIGS, shares=SHARES),
}

In [ ]:
print(f"{'example':26s} {'linear':>8s} {'k-NN':>8s} {'chance':>7s}   within class: linear / k-NN")


def within_score(entry, kind):
    # pooled within-class balanced accuracy, or "n/a" if that probe was skipped for the class
    res = entry["probes"].get(kind)
    return f"{res['pooled']['balanced_accuracy']:.2f}" if res else "n/a"


for name, (X_ex, d_ex, g_ex, y_ex, _) in examples.items():
    res = ev.assess_batch_effects(X_ex, d_ex, groups=g_ex, y=y_ex, n_splits=3, seed=0)
    lin, knn = (res["probes"][k]["pooled"] for k in ("linear", "knn"))
    within_txt = ", ".join(
        f"{c} {within_score(v, 'linear')} / {within_score(v, 'knn')}"
        for c, v in res["labels"]["within_class"].items()
        if v["status"] == "assessed"
    )
    print(
        f"{name:26s} {lin['balanced_accuracy']:8.2f} {knn['balanced_accuracy']:8.2f} "
        f"{lin['chance_balanced_accuracy']:7.2f}   {within_txt}"
    )

How to read this (pooled balanced accuracy on held-out recordings; chance = 1 / number of domains):

- **A:** close to chance, as expected when nothing was imposed. Near chance is not proof of no
  difference; a stronger probe or more data could still find one.
- **B:** domains are predictable overall *and* within each class: responses of the same class differ
  between domains.
- **C:** the domain is predictable overall (most clearly with k-NN) because the class mix differs,
  but within classes the scores are much closer to chance. Here, "correcting" domain predictability would mean erasing
  a real composition difference. The overall score alone would not have told you this.

The values come from finite synthetic samples, so they scatter a little around these patterns.

## 3. Your data

Replace **only the next cell** with your own arrays, keeping the names. Everything after it reads
`X`, `domains`, `groups`, `y` and (optionally) `y_true`, whatever your sample shape, domain IDs or
class names.

- `y` may contain `None` for unknown labels, or be `None` entirely: label-based sections are then skipped.
- `y_true` holds true labels for scoring if you have them (generated data only); otherwise leave it
  `None` and `y` is used.

In [ ]:
# ----- replace this cell with your own data -----------------------------------------------
X, domains, groups, y, y_true = simulate(
    7, RIGS, recordings=8, cells=40, technical=SHIFT, labelled=0.7
)
# X: (samples, time[, channels]); domains, groups: one ID per sample;
# y: labels with None for unknown (or y = None); y_true: true labels for scoring, or None
# ------------------------------------------------------------------------------------------

In [ ]:
X = np.asarray(X, dtype=float)
domains, groups = np.asarray(domains), np.asarray(groups)
y_eval = y_true if y_true is not None else y  # labels used only for scoring
domain_ids = sorted(set(domains.tolist()))
print("X:", X.shape, "| domains:", domain_ids)
for dom in domain_ids:
    rows = domains == dom
    print(f"  {dom!s:>10}: {rows.sum():5d} samples in {len(set(groups[rows])):3d} groups")
if y is not None:
    known = [v for v in y if v is not None]
    print("labels known for", len(known), "of", len(y), "samples; classes:", sorted(set(known)))

In [ ]:
# A look at the data: mean response of each domain (first channel), per class if labels exist.
traces = X.reshape(len(X), X.shape[1], -1)[..., 0]
classes_shown = sorted({v for v in y if v is not None})[:3] if y is not None else [None]
fig, axes = plt.subplots(
    1, len(classes_shown), figsize=(4 * len(classes_shown), 2.8), squeeze=False
)
for ax, cls in zip(axes[0], classes_shown, strict=True):
    for dom in domain_ids:
        rows = domains == dom
        if cls is not None:
            rows &= np.array([v == cls for v in y])
        if rows.any():
            ax.plot(traces[rows].mean(axis=0), label=str(dom))
    ax.set_title("all samples" if cls is None else f"class: {cls}")
    ax.set_xlabel("time (samples)")
axes[0][0].legend(fontsize=8)
fig.suptitle("Mean response per domain" + (" (synthetic)" if y_true is not None else ""))
fig.tight_layout()
plt.show()

## 4. Reserve a final test set of whole recordings

Before looking at any result, set aside whole recordings (groups) from **every** domain. Models are
trained only on the remaining development recordings; the final comparison uses the reserved ones,
and no setting is chosen by looking at them.

In [ ]:
TEST_GROUPS_PER_DOMAIN = 2


def hold_out_groups(domains, groups, per_domain, seed=0):
    """Pick `per_domain` whole groups per domain for testing; returns a boolean test mask."""
    rng = np.random.default_rng(seed)
    test = np.zeros(len(domains), dtype=bool)
    for dom in sorted(set(domains.tolist())):
        own = sorted(set(groups[domains == dom].tolist()))
        if len(own) < per_domain + 3:
            raise ValueError(f"domain {dom!r} has {len(own)} groups; need {per_domain + 3}.")
        test |= np.isin(groups, rng.choice(own, size=per_domain, replace=False))
    return test


is_test = hold_out_groups(domains, groups, TEST_GROUPS_PER_DOMAIN)
dev, test = ~is_test, is_test
X_dev, X_test = X[dev], X[test]
d_dev, d_test, g_dev, g_test = domains[dev], domains[test], groups[dev], groups[test]
y_dev = None if y is None else [v for v, m in zip(y, dev, strict=True) if m]
y_eval_test = None if y_eval is None else [v for v, m in zip(y_eval, test, strict=True) if m]
print(f"development: {dev.sum()} samples, {len(set(g_dev))} groups")
print(f"final test : {test.sum()} samples, {len(set(g_test))} groups (never used for decisions)")

## 5. Diagnose the development data

`assess_batch_effects` holds out whole groups (stratified group K-fold), checks that every domain is
in every training and test part, fits the probes' scaling on the training part only, and pools the
held-out predictions.

In [ ]:
diag = ev.assess_batch_effects(X_dev, d_dev, groups=g_dev, y=y_dev, n_splits=3, seed=0)
print("Question:", diag["question"], "\n")
for line in diag["messages"]:
    print("-", line)
print("\nLimitations:")
for line in diag["limitations"]:
    print("-", line)

In [ ]:
def plot_probe_scores(results, title):
    """Pooled held-out balanced accuracy per probe, with the chance level."""
    names = list(results)
    kinds = list(next(iter(results.values()))["probes"])
    fig, ax = plt.subplots(figsize=(1.6 + 1.3 * len(names), 3.2))
    width = 0.8 / len(kinds)
    for j, kind in enumerate(kinds):
        vals = [results[n]["probes"][kind]["pooled"]["balanced_accuracy"] for n in names]
        ax.bar(np.arange(len(names)) + (j - (len(kinds) - 1) / 2) * width, vals, width, label=kind)
    chance = next(iter(results.values()))["probes"][kinds[0]]["pooled"]["chance_balanced_accuracy"]
    ax.axhline(chance, color="k", ls="--", lw=1, label="chance")
    ax.set_xticks(np.arange(len(names)), names, rotation=20, ha="right")
    ax.set_ylim(0, 1.05)
    ax.set_ylabel("held-out balanced accuracy")
    ax.set_title(title)
    ax.legend(fontsize=8)
    fig.tight_layout()
    plt.show()


plot_probe_scores({"development data": diag}, "How well can the domain be predicted?")

## 6. Class composition and within-class predictability

Skipped automatically when `y` is `None`. Classes labelled in only one domain cannot be separated
from that domain; classes are assessed only where at least two domains have enough groups.

In [ ]:
labels = diag["labels"]
if labels is None:
    print("No labels: skipped.")
else:
    cls_names = labels["classes"]
    counts = np.array([[labels["counts"][d][c] for c in cls_names] for d in diag["domains"]])
    shares = counts / np.maximum(counts.sum(axis=1, keepdims=True), 1)
    fig, ax = plt.subplots(figsize=(5, 2.8))
    left = np.zeros(len(diag["domains"]))
    for j, c in enumerate(cls_names):
        ax.barh([str(d) for d in diag["domains"]], shares[:, j], left=left, label=str(c))
        left += shares[:, j]
    ax.set_xlabel("share of labelled samples")
    ax.set_title("Class composition per domain")
    ax.legend(fontsize=8, bbox_to_anchor=(1.02, 1), loc="upper left")
    fig.tight_layout()
    plt.show()
    print("unknown labels per domain:", labels["unknown_per_domain"])
    print("classes labelled in one domain only:", labels["confounded_classes"] or "none")
    for c, entry in labels["within_class"].items():
        if entry["status"] != "assessed":
            print(f"  {c}: skipped ({entry['reason']})")
            continue
        chance = next(iter(entry["probes"].values()))["pooled"]["chance_balanced_accuracy"]
        scores = ", ".join(
            f"{k} {v['pooled']['balanced_accuracy']:.2f}" for k, v in entry["probes"].items()
        )
        scores += f" (chance {chance:.2f}; domains {entry['domains']})"
        for kind, why in entry["skipped_probes"].items():
            scores += f"; {kind} skipped: {why}"
        print(f"  {c}: {scores}")

## 7. What does the evidence justify?

Use the numbers above together with what you know about the experiment:

- **Predictable overall and within classes** (as in example B): responses of the same class differ
  between domains. That is consistent with technical differences, but sub-types, animals or conditions
  could also differ. A correction is worth *trying*, with the checks below.
- **Predictable overall but not within classes** (as in example C): composition differs. Correcting
  would remove a real difference; usually analyse classes separately instead.
- **Classes labelled in only one domain:** class and domain cannot be disentangled for them.
- **Near chance:** no strong evidence of differences with this probe and design, which is not proof of
  absence.
- **Different biological sources** (e.g. somata versus axon terminals, different regions) need not
  become indistinguishable, and a method should not be judged by making them so.

The next sections show the correction workflow for the development data you have. Nothing is chosen by
looking at the final test set.

## 8. Fit a correction on development recordings only, with a zero-adversary control

`domain_weight=0` gives the same model without the adversary: the control that every comparison needs.
`RAVEPlus` uses the known labels (`None` = unknown); `VAEDANN` is an optional experimental alternative.
The data are standardized with development statistics (one mean and SD per feature, not per domain).

In [ ]:
mean, sd = X_dev.mean(axis=0), X_dev.std(axis=0) + 1e-8
Xs_dev, Xs_test = (X_dev - mean) / sd, (X_test - mean) / sd
EPOCHS = 60  # a short, fixed budget for this example; not tuned on the test set

models = {
    "RAVE, adversary off": RAVE(domain_weight=0.0, epochs=EPOCHS, seed=0),
    "RAVE": RAVE(epochs=EPOCHS, seed=0),
}
if RUN_RAVEPLUS and y_dev is not None:
    models["RAVE+, adversary off"] = RAVEPlus(domain_weight=0.0, epochs=EPOCHS, seed=0)
    models["RAVE+"] = RAVEPlus(epochs=EPOCHS, seed=0)
if RUN_VAEDANN:
    models["VAEDANN, adversary off"] = VAEDANN(domain_weight=0.0, seed=0)
    models["VAEDANN"] = VAEDANN(seed=0)

for name, model in models.items():
    uses_labels = isinstance(model, RAVEPlus) or (isinstance(model, VAEDANN) and y_dev is not None)
    if uses_labels:
        model.fit(Xs_dev, d_dev, y=y_dev)
    else:
        model.fit(Xs_dev, d_dev)
    print("fitted", name)

## 9. Compare on the same held-out recordings

The models never saw the test recordings. The diagnostic is run on the raw test data first; its
grouped splits are then reused for every representation, so all scores come from the same probe
training and test samples. Compare each model with its own adversary-off control. This is an
illustration of the comparison on synthetic data, not a ranking of methods, and nothing was tuned on
these test recordings.

In [ ]:
held_out = {"raw data": ev.assess_batch_effects(Xs_test, d_test, groups=g_test, n_splits=2, seed=0)}
test_splits = [(s["train"], s["test"]) for s in held_out["raw data"]["split"]["indices"]]
embeddings_dev, embeddings_test = {}, {}
for name, model in models.items():
    embeddings_dev[name], embeddings_test[name] = model.transform(Xs_dev), model.transform(Xs_test)
    held_out[name] = ev.assess_batch_effects(
        embeddings_test[name], d_test, splits=test_splits, seed=0
    )

for name, res in held_out.items():
    scores = ", ".join(
        f"{k} {v['pooled']['balanced_accuracy']:.2f}" for k, v in res["probes"].items()
    )
    print(f"{name:24s} {scores}")
plot_probe_scores(held_out, "Domain predictability on held-out recordings")

## 10. Reconstruction and biology on the same held-out samples

A lower domain score is only useful if responses are still reconstructed and biological information is
kept. Reconstruction skill is relative to predicting the development mean (1 = perfect, 0 = no better
than the mean). Biological prediction: a logistic classifier trained on the development representation
(labelled samples) predicts the test classes; for `RAVE+` its own class head is shown too. If class
scores are at their ceiling, as with the easy synthetic classes here, they cannot reveal a loss of
biological information; harder labels or other known response properties are more informative.

In [ ]:
recon, biology = {}, {}
for name, model in models.items():
    recon[name] = ev.reconstruction_scores(
        Xs_test, model.reconstruct(Xs_test), reference_mean=Xs_dev.mean(axis=0), groups=d_test
    )["reconstruction_skill"]

if y_dev is not None and y_eval_test is not None:
    known_dev = np.array([v is not None for v in y_dev])
    lab_dev = np.array([v for v in y_dev if v is not None])
    reps_dev = {"raw data": Xs_dev.reshape(len(Xs_dev), -1), **embeddings_dev}
    reps_test = {"raw data": Xs_test.reshape(len(Xs_test), -1), **embeddings_test}
    for name in reps_dev:
        clf = LogisticRegression(max_iter=2000).fit(reps_dev[name][known_dev], lab_dev)
        pred = clf.predict(reps_test[name])
        biology[name] = ev.classification_scores(y_eval_test, pred, groups=d_test)[
            "balanced_accuracy"
        ]
    for name, model in models.items():
        if isinstance(model, RAVEPlus):
            biology[name + " (own head)"] = ev.classification_scores(
                y_eval_test, model.predict(Xs_test)
            )["balanced_accuracy"]

print(f"{'representation':28s} {'recon. skill':>12s} {'class bal. acc.':>16s}")
for name in ["raw data", *models, *[n for n in biology if n.endswith("(own head)")]]:
    r = f"{recon[name]:12.2f}" if name in recon else f"{'-':>12s}"
    b = f"{biology[name]:16.2f}" if name in biology else f"{'-':>16s}"
    print(f"{name:28s} {r} {b}")

In [ ]:
# Illustration only: a 2-D view, fitted on development data. It is not a test for batch effects.
show = ["raw data"] + [n for n in ("RAVE, adversary off", "RAVE") if n in embeddings_dev]
reps_dev_all = {"raw data": Xs_dev.reshape(len(Xs_dev), -1), **embeddings_dev}
reps_test_all = {"raw data": Xs_test.reshape(len(Xs_test), -1), **embeddings_test}
fig, axes = plt.subplots(1, len(show), figsize=(3.6 * len(show), 3.2))
for ax, name in zip(np.atleast_1d(axes), show, strict=True):
    pca = PCA(n_components=2).fit(reps_dev_all[name])
    proj = pca.transform(reps_test_all[name])
    for dom in domain_ids:
        rows = d_test == dom
        ax.scatter(*proj[rows].T, s=6, alpha=0.6, label=str(dom))
    ax.set_title(name)
    ax.set_xticks([])
    ax.set_yticks([])
np.atleast_1d(axes)[0].legend(fontsize=8, markerscale=2)
fig.suptitle("Held-out samples, first two principal components (fitted on development data)")
fig.tight_layout()
plt.show()

## 11. Summary and how to use your own data

- Run `assess_batch_effects` **before** correcting, with `groups` set to your independent units
  (usually recordings). Read the overall score, the within-class scores and the composition table
  together.
- Decide whether correction is appropriate (section 7). A predictable domain is not by itself a
  reason to correct.
- If you correct, train models on development recordings only, always with a `domain_weight=0`
  control, and compare on held-out recordings with **the same splits** (`splits=`), alongside
  reconstruction and biological scores.
- Choose settings on development or validation data, never on the final test set.

To use your own recordings, replace the cell marked *replace this cell with your own data* in
section 3. More detail: the [quickstart](quickstart.ipynb), the
[model comparison notebook](compare_methods.ipynb) and the
[README](https://github.com/dgonschorek/neuro_rave#check-the-result-on-your-own-data).